In [ ]:
using Pkg
Pkg.activate(".")
Pkg.develop(path="..")

using Revise

In [ ]:
if success(`nvidia-smi`)
    println("CUDA is available. Loading CUDA.jl...")
    using CUDA
end

In [ ]:
using bslLD, Plots, Statistics

bslLD.use_cuda!()

In [ ]:
grid =  bslLD.Grid([0.0,-4.0,-4.0],[60.0,4.0,4.0],[64,33,33],1, 1.0, 1)

simTime = bslLD.SimulationTime(0.02, 10000, gyro_frequency=1.0)

# initFuncv(v)= exp(-(v+2)^2 / 2) / sqrt(2*pi)+ exp(-(v-2)^2 / 2) / sqrt(2*pi)
initFuncv(v) = exp(-v^2 / 2) / sqrt(2*pi)
initFuncx(x) = 1+ 0.000001 * rand()
f = bslLD.Distribution(grid, 0.5,initFuncv=initFuncv, initFuncx=initFuncx);
e = bslLD.empty_vectorfield(grid);


In [ ]:
#2D3V grid with 32x8 32x32x32 grid points 
grid = bslLD.Grid([0.0,0.0, -4.0,-4.0,-4.0],[20.0,20.0, 4.0,4.0,4.0],[32,32,33,33,33],2, 1.0, 2)
simTime = bslLD.SimulationTime(0.02, 10000, gyro_frequency=1.0)


In [ ]:
f = bslLD.Distribution(grid, 0.5);


In [ ]:
mutable struct Diag
    rho::Vector
    Ex::Vector
end
Diag() = Diag([], [])

function diags!(diags, f, rho, Ex, grid)
    push!(diags.rho, copy(rho.data[:]))
    push!(diags.Ex, copy(Ex.data[:]))
end

function step!(f, grid, simTime)
    bslLD.advectX!(f, grid, simTime)
    rho = bslLD.compute_density(f, grid)
    sol = bslLD.solve_fields(bslLD.Moments(rho), grid, bslLD.AdiabaticFieldSolver())
    bslLD.advectV!(f, grid, simTime, sol.E)
end


In [ ]:
step!(f, grid, simTime, Diag())

In [ ]:
@time step!(f, grid, simTime, Diag())

In [ ]:
@time step!(f, grid, simTime, plan)

In [ ]:
diags = Diag()
while bslLD.continue_advection(simTime)
    step!(f, grid, simTime, plan, diags)
    bslLD.advance!(simTime)
end    

In [ ]:
using FFTW, DSP

In [ ]:
locData = transpose(hcat(map(x-> x.-mean(x), Array.(diags.rho))...))


Nx, Ny = size(locData)
w = kaiser(Ny, 6)

windowed = locData .* w'        # broadcast along second dim (1 × Ny)

heatmap(log.(abs.(fft(windowed))[1:200,1:round(Int,Ny/2)]))

In [ ]:

function measureStep(backendFunc,grid)
    backendFunc()

    f = bslLD.Distribution(grid, 0.5,initFuncv=initFuncv, initFuncx=initFuncx);
    e = bslLD.empty_vectorfield(grid);

    [step!(f, grid, Diag()) for _ in 1:10]   

    @time [step!(f, grid, Diag()) for _ in 1:10]
end

grid =  bslLD.Grid([0.0,-4.0,-4.0],[60.0,4.0,4.0],[256,257,257],0.02,10000,1, 1.0, 1)

measureStep(() -> bslLD.use_cuda!(),grid)
measureStep(() -> bslLD.use_cpu!(), grid)


#   1.270276 seconds (278.26 k allocations: 8.469 MiB, 1.77% gc time, 64.35% compilation time)
# 247.798143 seconds (46.79 k allocations: 29.209 GiB, 84.31% gc time, 0.89% compilation time)
#   0.059636 seconds (60.53 k allocations: 2.844 MiB, 20.65% gc time, 61.27% compilation time)
#   0.126698 seconds (46.27 k allocations: 83.402 MiB, 28.70% compilation time)





In [ ]:
#   0.018241 seconds (22.21 k allocations: 1.417 MiB, 22.05% gc time)
#  16.719033 seconds (2.88 k allocations: 1.254 GiB, 88.55% gc time)


grid =  bslLD.Grid([0.0,-4.0],[60.0,4.0],[1024,1024],0.02,10000,1, 1.0, 1)

measureStep(() -> bslLD.use_cuda!(),grid)
measureStep(() -> bslLD.use_cpu!(), grid)



In [ ]:
grid =  bslLD.Grid([0.0,-4.0,-4.0],[60.0,4.0,4.0],[256,255,255],0.02,10000,1, 1.0, 1)
f = bslLD.Distribution(grid, 0.5);

plan = bslLD.AdvectionPlan(f, grid)


In [ ]:
size(f.data)

In [ ]:
bslLD.use_cpu!()

In [ ]:
function compare_allocations(backendFunc)
    backendFunc()
    grid =  bslLD.Grid([0.0,-4.0,-4.0],[60.0,4.0,4.0],[256,257,257],0.02,10000,1, 1.0, 1)
    f = bslLD.Distribution(grid, 0.5);
    e = bslLD.empty_vectorfield(grid);

    plan = bslLD.AdvectionPlan(f, grid)

    #Warmup
    bslLD.advectX!(f,grid, plan)
    bslLD.advectX!(f,grid)
    bslLD.advectV!(f,grid,e)
    bslLD.advectV!(f,grid,e, plan)

    println("Measuring allocations for ", typeof(f.data))
    println("AdvectX with plan:")
    @time bslLD.advectX!(f,grid, plan)
    println("AdvectX without plan:")
    @time bslLD.advectX!(f,grid)
    println("AdvectV with plan:")
    @time bslLD.advectV!(f,grid,e, plan)
    println("AdvectV without plan:")
    @time bslLD.advectV!(f,grid,e)

    println(" ")

    println("Check for type stability:")
    println("AdvectX with plan:")
    @code_warntype bslLD.advectX!(f,grid, plan)
    println("AdvectX without plan:")
    @code_warntype bslLD.advectX!(f,grid)
    println("AdvectV with plan:")
    @code_warntype bslLD.advectV!(f,grid,e, plan)
    println("AdvectV without plan:")
    @code_warntype bslLD.advectV!(f,grid,e)
end

In [ ]:
compare_allocations(() -> bslLD.use_cuda!())


In [ ]:
compare_allocations(() -> bslLD.use_cpu!())

In [ ]:
CUDA.@profile bslLD.advectX!(f, grid, plan)

In [ ]:
CUDA.@profile bslLD.advectV!(f, grid, e, plan)

In [ ]:
print("Size of f.data: ")
println(size(f.data))
CUDA.@profile bslLD.advectV!(f, grid, e, plan)

In [ ]:

bslLD.use_cuda!()

grid =  bslLD.Grid([0.0,-4.0,-4.0],[60.0,4.0,4.0],[256,257,257],0.02,10000,1, 1.0, 1)
f = bslLD.Distribution(grid, 0.5);
e = bslLD.empty_vectorfield(grid);

plan = bslLD.AdvectionPlan(f, grid)


@code_warntype bslLD._advect_x_planned!(f, grid, plan, CUDABackend())
@code_warntype bslLD._advect_v_planned!(f, grid, e, plan, CUDABackend())

In [ ]:
@allocated ntuple(d -> e[d].data, Val(2))
#6000

In [ ]:
arr = f.data

In [ ]:
@allocated kernel!(arr, ctx; ndrange=length(arr))

In [ ]:

bslLD.use_cpu!()

grid =  bslLD.Grid([0.0,-4.0,-4.0],[60.0,4.0,4.0],[256,257,257],0.02,10000,1, 1.0, 1)
f = bslLD.Distribution(grid, 0.5);
e = bslLD.empty_vectorfield(grid);

plan = bslLD.AdvectionPlan(f, grid)


@code_warntype bslLD._advect_x_planned!(f, grid, plan, bslLD.KernelAbstractions.CPU())
@code_warntype bslLD._advect_v_planned!(f, grid, e, plan, bslLD.KernelAbstractions.CPU())

In [ ]:
plan.backend